In [6]:
import jax
import jax.numpy as jnp
from jax import random

jax.config.update("jax_enable_x64", True) # Crucial for higher-order derivatives

# 1. Define the Metric (The Statistical Prior/Topology)
def gaussian_metric(theta):
    """
    Fisher Information Matrix for a univariate Gaussian.
    theta = [mu, sigma]
    """
    mu, sigma = theta[0], theta[1]
    return jnp.array([
        [1.0 / sigma**2, 0.0],
        [0.0, 2.0 / sigma**2]
    ])

# 2. Differential Geometry Operators
def christoffel_symbols(metric_fn, theta):
    """Computes the Levi-Civita connection coefficients \Gamma^k_{ij}."""
    # Derivative of metric: dg[i, j, k] = \partial_k g_{ij}
    dg = jax.jacfwd(metric_fn)(theta)
    g = metric_fn(theta)
    g_inv = jnp.linalg.inv(g)
    
    # \Gamma^k_{ij} = 0.5 * g^{kl} (dg_{li, j} + dg_{lj, i} - dg_{ij, l})
    # Using einsum for tensor contractions
    term1 = jnp.einsum('kl,lij->kij', g_inv, dg)
    term2 = jnp.einsum('kl,lji->kij', g_inv, dg)
    term3 = jnp.einsum('kl,ijl->kij', g_inv, dg)
    
    gamma = 0.5 * (term1 + term2 - term3)
    return gamma

def riemann_tensor(metric_fn, theta):
    """Computes the Riemann curvature tensor R^l_{ijk}."""
    gamma_fn = lambda t: christoffel_symbols(metric_fn, t)
    gamma = gamma_fn(theta)
    
    # dgamma[l, i, j, m] = \partial_m \Gamma^l_{ij}
    dgamma = jax.jacfwd(gamma_fn)(theta)
    
    # R^l_{ijk} = \partial_j \Gamma^l_{ik} - \partial_k \Gamma^l_{ij} 
    #             + \Gamma^l_{jm}\Gamma^m_{ik} - \Gamma^l_{km}\Gamma^m_{ij}
    
    # \partial_j \Gamma^l_{ik} -> dgamma[l, i, k, j]
    term1 = jnp.einsum('likj->lijk', dgamma)
    
    # \partial_k \Gamma^l_{ij} -> dgamma[l, i, j, k]
    term2 = dgamma 
    
    # \Gamma^l_{jm}\Gamma^m_{ik} -> sum_m gamma[l, j, m] * gamma[m, i, k]
    term3 = jnp.einsum('ljm,mik->lijk', gamma, gamma)
    
    # \Gamma^l_{km}\Gamma^m_{ij} -> sum_m gamma[l, k, m] * gamma[m, i, j]
    term4 = jnp.einsum('lkm,mij->lijk', gamma, gamma)
    
    return term1 - term2 + term3 - term4
    
def ricci_and_scalar(metric_fn, theta):
    """Computes the Ricci tensor R_{ij} and Scalar Curvature R."""
    R_lijk = riemann_tensor(metric_fn, theta)
    g = metric_fn(theta)
    g_inv = jnp.linalg.inv(g)
    
    # Ricci tensor R_{ik} = R^l_{ilk}
    ricci_tensor = jnp.einsum('lilk->ik', R_lijk)
    
    # Scalar curvature R = g^{ik} R_{ik}
    scalar_curvature = jnp.einsum('ik,ik->', g_inv, ricci_tensor)
    
    return ricci_tensor, scalar_curvature

# 3. Model Scoring via Exact Evidence Expansion
def geometric_model_score(theta_mle, metric_fn, n_samples, log_likelihood, k_params):
    """
    Evaluates the truncated expansion: -log Z_M(n)
    """
    ricci, R = ricci_and_scalar(metric_fn, theta_mle)
    
    # 1. Classical BIC terms
    bic_penalty = (k_params / 2.0) * jnp.log(n_samples / (2 * jnp.pi))
    
    # 2. Robustness/Naturalness (Volume normalization)
    # det(J) / det(I) approximation - assuming empirical J ~ Fisher I at MLE for flat data
    robustness = 0.0 
    
    # 3. O(n^-1) Curvature Penalty (Using intrinsic R)
    # T_ij (Stress-Energy from data) is assumed isolated in the residual variance here.
    geom_penalty = R / n_samples
    
    # Total Score (Lower is better)
    score = -log_likelihood + bic_penalty + robustness + geom_penalty
    
    return score, R

def gmrf_precision(theta, L):
    """Constructs the precision matrix Q = alpha*I + beta*L."""
    alpha, beta = theta[0], theta[1]
    I = jnp.eye(L.shape[0])
    return alpha * I + beta * L

def gmrf_metric(theta, L):
    """
    Computes the 2x2 Fisher Information Metric for the GMRF.
    theta = [alpha, beta]
    """
    Q = gmrf_precision(theta, L)
    Q_inv = jnp.linalg.inv(Q)
    
    # Partial derivatives of Q
    dQ_dalpha = jnp.eye(L.shape[0])
    dQ_dbeta = L
    
    # Compute the Fisher metric components using the exact trace formula
    g11 = 0.5 * jnp.trace(Q_inv @ dQ_dalpha @ Q_inv @ dQ_dalpha)
    g12 = 0.5 * jnp.trace(Q_inv @ dQ_dalpha @ Q_inv @ dQ_dbeta)
    g22 = 0.5 * jnp.trace(Q_inv @ dQ_dbeta @ Q_inv @ dQ_dbeta)
    
    return jnp.array([
        [g11, g12],
        [g12, g22]
    ])

# --- Let's test it on a simple 1D Cycle Graph (Ring Network) ---
N = 10 # Number of nodes
# Construct a simple ring Laplacian (1D discrete continuum)
off_diag = jnp.diag(jnp.ones(N-1), 1) + jnp.diag(jnp.ones(N-1), -1)
off_diag = off_diag.at[0, N-1].set(1.0).at[N-1, 0].set(1.0) # Periodic boundary
L_ring = jnp.diag(jnp.sum(off_diag, axis=1)) - off_diag

# Define a wrapped metric function to pass to the geometry engine
metric_fn_gmrf = lambda t: gmrf_metric(t, L_ring)

# Evaluate at some arbitrary structural state (alpha=1.0, beta=0.5)
theta_gmrf = jnp.array([1.0, 0.5])

# (Assuming ricci_and_scalar is defined from the previous script)
ricci_tensor, scalar_R = ricci_and_scalar(metric_fn_gmrf, theta_gmrf)

print(f"Graph Size: {N} nodes")
print(f"GMRF Scalar Curvature (R): {scalar_R:.6f}")


# --- 1. Simulate "True" Data (The Mass) ---
def simulate_empirical_covariance(N, true_theta, L, n_samples, key):
    """Generates an empirical covariance matrix S from a true GMRF."""
    Q_true = gmrf_precision(true_theta, L)
    Cov_true = jnp.linalg.inv(Q_true)
    
    # Draw samples from N(0, Cov_true)
    samples = random.multivariate_normal(
        key, 
        mean=jnp.zeros(N), 
        cov=Cov_true, 
        shape=(n_samples,)
    )
    # Compute empirical covariance S
    S = jnp.cov(samples, rowvar=False)
    return S

# --- 2. The Stress-Energy Tensor ---
def empirical_stress_energy(theta, L, S):
    """Computes T_ij = U_i U_j from the empirical covariance S."""
    Q = gmrf_precision(theta, L)
    Q_inv = jnp.linalg.inv(Q)
    
    # Partial derivatives (from previous logic)
    dQ_dalpha = jnp.eye(L.shape[0])
    dQ_dbeta = L
    
    # The Score Vector U_i
    residual = Q_inv - S
    U_alpha = 0.5 * jnp.trace(residual @ dQ_dalpha)
    U_beta = 0.5 * jnp.trace(residual @ dQ_dbeta)
    
    U = jnp.array([U_alpha, U_beta])
    
    # T_ij is the outer product of the Score
    T_ij = jnp.outer(U, U)
    return T_ij


# --- 1. The Pull of Gravity (Score Vector) ---
def compute_score(theta, L, S):
    """Computes the Score vector U_i from the empirical covariance S."""
    Q = gmrf_precision(theta, L)
    Q_inv = jnp.linalg.inv(Q)
    
    dQ_dalpha = jnp.eye(L.shape[0])
    dQ_dbeta = L
    
    # U_i = 0.5 * Tr((Q^-1 - S) dQ/dtheta_i)
    # This is the exact gradient of the Gaussian log-likelihood
    residual = Q_inv - S
    U_alpha = 0.5 * jnp.trace(residual @ dQ_dalpha)
    U_beta = 0.5 * jnp.trace(residual @ dQ_dbeta)
    
    return jnp.array([U_alpha, U_beta])

# --- 2. The Forced Ricci Step ---
def forced_ricci_step(theta, L, S, kappa):
    """
    Executes one time-step of the flow.
    Projects the metric flow onto parameter space via the inverse metric.
    """
    # 1. Compute current spacetime metric
    g = gmrf_metric(theta, L)
    g_inv = jnp.linalg.inv(g)
    
    # 2. Compute the gravitational pull from the data mass
    U = compute_score(theta, L, S)
    
    # 3. Update parameters along the geodesic
    delta_theta = kappa * (g_inv @ U)
    return theta + delta_theta

# --- Execution: The Integration Loop ---
# (Assuming L_ring and S_empirical are in memory from the previous script)
current_theta = jnp.array([2.0, 0.1]) # Our wrong guess
kappa = 0.05 # The coupling constant (time-step size)
n_steps = 15

print(f"Target True Parameters : alpha=1.000, beta=0.500")
print(f"Initial State (t=0)    : alpha={current_theta[0]:.3f}, beta={current_theta[1]:.3f}\n")

for step in range(1, n_steps + 1):
    current_theta = forced_ricci_step(current_theta, L_ring, S_empirical, kappa)
    
    if step % 3 == 0 or step == n_steps:
        # Recompute T_ij just to watch the forces decay
        T_tensor = empirical_stress_energy(current_theta, L_ring, S_empirical)
        stress_magnitude = jnp.linalg.norm(T_tensor)
        print(f"Step {step:2d} | alpha={current_theta[0]:.4f}, beta={current_theta[1]:.4f} | Stress Magnitude: {stress_magnitude:.4f}")

# --- Execution: Observing the Warping ---
key = random.PRNGKey(42)
N = 10
n_samples = 50

# 1. Generate the Laplacian (from the previous script)
off_diag = jnp.diag(jnp.ones(N-1), 1) + jnp.diag(jnp.ones(N-1), -1)
off_diag = off_diag.at[0, N-1].set(1.0).at[N-1, 0].set(1.0)
L_ring = jnp.diag(jnp.sum(off_diag, axis=1)) - off_diag

# 2. Simulate reality: S
true_theta = jnp.array([1.0, 0.5]) 
S_empirical = simulate_empirical_covariance(N, true_theta, L_ring, n_samples, key)

# 3. Initialize our model at a WRONG state (empty vacuum)
guess_theta = jnp.array([2.0, 0.1]) 

# 4. Compute the forces
T_tensor = empirical_stress_energy(guess_theta, L_ring, S_empirical)

print(f"Current Guess: alpha={guess_theta[0]}, beta={guess_theta[1]}")
print(f"Empirical Stress-Energy Tensor (T_ij):\n{T_tensor}")
# --- Execution ---
#theta_mle = jnp.array([0.0, 1.0]) # Standard Normal MLE
#ricci_tensor, scalar_R = ricci_and_scalar(gaussian_metric, theta_mle)

#print(f"Ricci Tensor:\n{ricci_tensor}")
#print(f"Scalar Curvature (R): {scalar_R:.5f}")

Graph Size: 10 nodes
GMRF Scalar Curvature (R): -0.000000
Target True Parameters : alpha=1.000, beta=0.500
Initial State (t=0)    : alpha=2.000, beta=0.100

Step  3 | alpha=1.7734, beta=0.1908 | Stress Magnitude: 0.3380
Step  6 | alpha=1.6087, beta=0.2608 | Stress Magnitude: 0.2479
Step  9 | alpha=1.4849, beta=0.3163 | Stress Magnitude: 0.1818
Step 12 | alpha=1.3896, beta=0.3609 | Stress Magnitude: 0.1334
Step 15 | alpha=1.3149, beta=0.3974 | Stress Magnitude: 0.0979
Current Guess: alpha=2.0, beta=0.1
Empirical Stress-Energy Tensor (T_ij):
[[ 0.25266452 -0.22941764]
 [-0.22941764  0.20830963]]


In [7]:
import jax
import jax.numpy as jnp

# --- 1. The Geometric Gradient (Dark Energy) ---
def compute_curvature_gradient(theta, metric_fn):
    """
    Computes \partial_j R, the gradient of the scalar curvature.
    """
    # Wrapper to return only the scalar curvature R for autodiff
    def just_scalar_R(t):
        _, R = ricci_and_scalar(metric_fn, t)
        return R
    
    # jax.grad computes the exact gradient vector [\partial_alpha R, \partial_beta R]
    grad_R = jax.grad(just_scalar_R)(theta)
    return grad_R

# --- 2. The Regularized Field Equations ---
def regularized_ricci_step(theta, L, S, kappa, n_samples):
    """
    Executes one time-step of the curvature-regularized flow.
    \theta_{t+1}^i = \theta_t^i + \kappa g^{ij} (U_j - (1/n) \partial_j R)
    """
    # 1. Compute current spacetime metric and its inverse
    metric_fn = lambda t: gmrf_metric(t, L)
    g = metric_fn(theta)
    g_inv = jnp.linalg.inv(g)
    
    # 2. Compute the gravitational pull from the empirical data
    U = compute_score(theta, L, S)
    
    # 3. Compute the geometric pushback (Gradient of Scalar Curvature)
    grad_R = compute_curvature_gradient(theta, metric_fn)
    
    # 4. Assemble the total force vector (Data Pull - Geometric Pushback)
    # The 1/n scaling controls the strength of the geometric regularizer
    total_force = U - (1.0 / n_samples) * grad_R
    
    # 5. Update parameters along the regularized geodesic
    delta_theta = kappa * (g_inv @ total_force)
    return theta + delta_theta

# --- Execution: The Regularized Integration Loop ---
# (Assuming L_ring, S_empirical, kappa are loaded from previous blocks)
current_theta = jnp.array([2.0, 0.1])
n_samples = 15 # Small n to amplify the geometric regularizer

print("--- Curvature-Regularized Optimization ---")
for step in range(1, 16):
    current_theta = regularized_ricci_step(current_theta, L_ring, S_empirical, kappa, n_samples)
    
    if step % 3 == 0 or step == 15:
        # We can observe the active gradient of R fighting the Score U
        metric_fn = lambda t: gmrf_metric(t, L_ring)
        grad_R = compute_curvature_gradient(current_theta, metric_fn)
        U = compute_score(current_theta, L_ring, S_empirical)
        
        print(f"Step {step:2d} | alpha={current_theta[0]:.4f}, beta={current_theta[1]:.4f}")
        print(f"         | Score U: [{U[0]:.4f}, {U[1]:.4f}]")
        print(f"         | Grad R : [{grad_R[0]:.4f}, {grad_R[1]:.4f}]")

--- Curvature-Regularized Optimization ---
Step  3 | alpha=1.7734, beta=0.1908
         | Score U: [-0.4274, 0.3941]
         | Grad R : [-0.0000, -0.0000]
Step  6 | alpha=1.6087, beta=0.2608
         | Score U: [-0.3640, 0.3396]
         | Grad R : [0.0000, 0.0000]
Step  9 | alpha=1.4849, beta=0.3163
         | Score U: [-0.3105, 0.2923]
         | Grad R : [0.0000, 0.0000]
Step 12 | alpha=1.3896, beta=0.3609
         | Score U: [-0.2650, 0.2514]
         | Grad R : [0.0000, 0.0000]
Step 15 | alpha=1.3149, beta=0.3974
         | Score U: [-0.2264, 0.2160]
         | Grad R : [-0.0000, -0.0000]


In [8]:
import jax
import jax.numpy as jnp
from jax.scipy.stats import norm

# --- 1. The Marginal Log-Likelihood ---
def mixture_log_pdf(x, theta):
    """Computes the marginal log p(x|theta) for a single observation."""
    mu1, mu2, sigma1, sigma2, w1 = theta[0], theta[1], theta[2], theta[3], theta[4]
    w2 = 1.0 - w1
    
    # We add a tiny epsilon to prevent log(0) near boundaries
    p1 = w1 * norm.pdf(x, loc=mu1, scale=sigma1)
    p2 = w2 * norm.pdf(x, loc=mu2, scale=sigma2)
    return jnp.log(p1 + p2 + 1e-12)

# --- 2. The Highly Curved Fisher Metric ---
def marginal_fisher_metric(theta):
    """
    Computes the exact Fisher Information Metric numerically using 1D quadrature.
    g_ij = \int [ \partial_i \log p(x|\theta) \partial_j \log p(x|\theta) ] p(x|\theta) dx
    """
    # Define a dense grid over the support of the distribution to integrate over
    # (Dynamically sized based on current means/stdevs for accuracy)
    span = jnp.max(jnp.array([theta[2], theta[3]])) * 5.0
    center = (theta[0] + theta[1]) / 2.0
    x_grid = jnp.linspace(center - span, center + span, 500)
    dx = x_grid[1] - x_grid[0]
    
    # JAX magic: Vectorized gradient of the log_pdf w.r.t theta, evaluated at all x
    grad_fn = jax.vmap(jax.grad(mixture_log_pdf, argnums=1), in_axes=(0, None))
    grads = grad_fn(x_grid, theta)  # Shape: (500, 5)
    
    # Compute the actual density p(x|\theta) to weight the expectation
    log_p_x = jax.vmap(mixture_log_pdf, in_axes=(0, None))(x_grid, theta)
    p_x = jnp.exp(log_p_x)
    
    # g_ij = sum_x (grad_i * grad_j * p(x) * dx)
    fisher = jnp.einsum('ni,nj,n->ij', grads, grads, p_x) * dx
    
    # Add a tiny ridge penalty to the diagonal to ensure invertibility during flow
    return fisher + jnp.eye(5) * 1e-6

In [10]:
import jax
import jax.numpy as jnp
from jax import random

# --- 1. Simulate Overlapping GMM Data ---
def simulate_gmm_data(key, n_samples, true_theta):
    """Draws samples from a 2-component Gaussian mixture."""
    mu1, mu2, sig1, sig2, w1 = true_theta
    
    key_z, key_x1, key_x2 = random.split(key, 3)
    # Latent assignments
    z = random.bernoulli(key_z, p=w1, shape=(n_samples,))
    
    # Draw from both components
    x1 = random.normal(key_x1, shape=(n_samples,)) * sig1 + mu1
    x2 = random.normal(key_x2, shape=(n_samples,)) * sig2 + mu2
    
    # Mix based on assignments
    x = jnp.where(z, x1, x2)
    return x

# --- 2. The Score Vector for GMM ---
def gmm_score(theta, data):
    """Computes the empirical gradient of the log-likelihood."""
    # Sum of gradients over all data points
    grad_fn = jax.vmap(jax.grad(mixture_log_pdf, argnums=1), in_axes=(0, None))
    grads = grad_fn(data, theta)
    return jnp.sum(grads, axis=0)

# --- 3. The Execution Loop ---
key = random.PRNGKey(101)
n_samples = 40 # Small n amplifies the geometric pushback

# True parameters: Two clearly separated components
# [mu1, mu2, sigma1, sigma2, w1]
true_theta = jnp.array([-2.0, 2.0, 1.0, 1.0, 0.5])
data = simulate_gmm_data(key, n_samples, true_theta)

# Initial Guess: Dangerously close to the singularity (mu1 ~= mu2)
# The likelihood alone might merge these, but the geometry will fight it.
current_theta = jnp.array([-0.1, 0.1, 1.2, 1.2, 0.5])
kappa = 0.01 # Careful learning rate for curved space

print("--- Curvature-Regularized GMM Flow ---")
print(f"Target : mu1={true_theta[0]:.2f}, mu2={true_theta[1]:.2f}")
print(f"Initial: mu1={current_theta[0]:.2f}, mu2={current_theta[1]:.2f}\n")

for step in range(1, 11):
# 1. Spacetime Metric with Stronger Singularity Dampening
    g = marginal_fisher_metric(current_theta)
    
    # Add a Levenberg-Marquardt style ridge to bound the inverse metric
    # This acts as a 'minimum distance' cut-off from the singularity
    g_inv = jnp.linalg.inv(g + jnp.eye(5) * 1e-2) 
    
    # 2. Gravitational Pull (Data)
    U = gmm_score(current_theta, data)
    
    # 3. Geometric Pushback (Dark Energy)
    grad_R = compute_curvature_gradient(current_theta, marginal_fisher_metric)
    
    # 4. Total Force
    total_force = U - (1.0 / n_samples) * grad_R
    
    # 5. Geodesic Update with Trust Region (Clipping)
    natural_step = g_inv @ total_force
    step_norm = jnp.linalg.norm(natural_step)
    
    # The 'speed of light' constraint: never jump more than 0.5 units per step
    max_step_size = 0.5 
    safe_step = jnp.where(step_norm > max_step_size, 
                          natural_step * (max_step_size / step_norm), 
                          natural_step)
    
    current_theta = current_theta + kappa * safe_step
    if step % 2 == 0:
        print(f"Step {step:2d} | mu1={current_theta[0]:.3f}, mu2={current_theta[1]:.3f}")
        print(f"        | Score U (mu1, mu2) : [{U[0]:.2f}, {U[1]:.2f}]")
        print(f"        | Grad R  (mu1, mu2) : [{grad_R[0]:.2f}, {grad_R[1]:.2f}]")

--- Curvature-Regularized GMM Flow ---
Target : mu1=-2.00, mu2=2.00
Initial: mu1=-0.10, mu2=0.10

Step  2 | mu1=-0.093, mu2=0.093
        | Score U (mu1, mu2) : [-3.00, 1.56]
        | Grad R  (mu1, mu2) : [-41324114.92, 41324114.92]
Step  4 | mu1=-0.086, mu2=0.086
        | Score U (mu1, mu2) : [-2.83, 1.40]
        | Grad R  (mu1, mu2) : [-32845086.36, 32845086.36]
Step  6 | mu1=-0.079, mu2=0.079
        | Score U (mu1, mu2) : [-2.68, 1.24]
        | Grad R  (mu1, mu2) : [-25991775.08, 25991775.08]
Step  8 | mu1=-0.074, mu2=0.074
        | Score U (mu1, mu2) : [-2.48, 1.06]
        | Grad R  (mu1, mu2) : [-24781339.44, 24781339.44]
Step 10 | mu1=-0.069, mu2=0.069
        | Score U (mu1, mu2) : [-2.40, 0.96]
        | Grad R  (mu1, mu2) : [-17577728.06, 17577728.06]
